In [ ]:
!pip install transformers datasets evaluate seqeval

In [ ]:
! cd data & git clone https://github.com/UniversalNER/UNER_Slovak-SNK
! cd data & git clone https://github.com/UniversalNER/UNER_English-EWT
! cd data & git clone https://github.com/UniversalNER/UNER_Swedish-Lines
! cd data & git clone https://github.com/UniversalNER/UNER_Norwegian-NDT
! cd data & git clone https://github.com/UniversalNER/UNER_Hebrew-HTB
! cd data & git clone https://github.com/UniversalNER/UNER_Romanian-LegalNERo
! cd data & git clone https://github.com/UniversalNER/UNER_Portuguese-Bosque
! cd data & git clone https://github.com/UniversalNER/UNER_German-PUD
! cd data & git clone https://github.com/UniversalNER/UNER_Chinese-GSD
! cd data & git clone https://github.com/UniversalNER/UNER_Croatian-SET
! cd data & git clone https://github.com/UniversalNER/UNER_Serbian-SET
! cd data & git clone https://github.com/UniversalNER/UNER_Danish-DDT

In [3]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, DataCollatorForTokenClassification, AutoConfig
import torch
from tqdm.auto import tqdm
from datasets import Dataset
import evaluate


In [4]:
torch.cuda.is_available()

True

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
from pathlib import Path

def load_iob(file_path: str | Path) -> tuple[list[str], list[str]]:
    with open(file_path, "r", encoding="UTF-8") as f:
        lines = f.readlines()
    words = list()
    tags = list()
    for line in lines:
        if line[0] != "#" and line != "\n":
            _, word, tag, _, _ = line.strip().split("\t")
            words.append(word)
            tags.append(tag)
    return words, tags
    
tag_to_idx = {"O": 0, 'B-ORG': 1, 'I-ORG': 2, 'I-PER': 3, 'B-PER': 4, 'B-LOC': 5, 'I-LOC': 6, }
idx_to_tag = {i: tag for tag, i in tag_to_idx.items()}


In [ ]:
MULTI_MODEL_NAME = "google-bert/bert-base-multilingual-cased"
LR = 2e-5
EPOCHS = 3
BATCH_SIZE = 8

In [ ]:
def align_labels(tokenized, labels):
    # 2) Prepare a new "labels" list aligned to the subword tokens
    all_labels = torch.empty_like(tokenized["input_ids"], dtype=torch.int8)
    for i in range(all_labels.shape[0]):
        word_ids = tokenized.word_ids(batch_index = i)

        prev_word_id = None

        for j, word_id in enumerate(word_ids):
            if word_id is None:
                # e.g. special tokens or padding
                all_labels[i, j] = -100
            elif word_id == prev_word_id:
                # subword token of the same word => ignore
                all_labels[i, j] = -100
            else:
                # new subword, so use the label for the original token
                all_labels[i, j] = tag_to_idx[labels[word_id]]

            prev_word_id = word_id


    # 3) Attach the new "labels" to our tokenized inputs

    tokenized["labels"] = all_labels

    # 4) Return the updated dictionary
    return tokenized

In [14]:
def load_tokenize_align(file_path: str | Path, model: str) -> Dataset:
    words, tags = load_iob(file_path)
    tokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)
    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        max_length=128,
        truncation="only_first", # Keep the beginning, slice the rest
        return_overflowing_tokens=True, # Returns a list of chunks
        padding="max_length",    # Ensure all chunks are exactly 128
        return_tensors="pt"      # Returns PyTorch tensors (or "tf", "np")
    )
    tokenized = align_labels(tokenized, tags)

    return Dataset.from_dict({k: v for k, v in tokenized.items()})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MULTI_MODEL_NAME, use_fast=True)
multi_config = AutoConfig.from_pretrained(MULTI_MODEL_NAME, num_labels=len(tag_to_idx))
multi_model = AutoModelForTokenClassification.from_pretrained(
    MULTI_MODEL_NAME,
    config=multi_config
)
# Create a data collator
multi_data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
from torch.utils.data import DataLoader

english_train = load_tokenize_align("UNER_English-EWT/en_ewt-ud-train.iob2", MULTI_MODEL_NAME)
slovak_train = load_tokenize_align("UNER_Slovak-SNK/sk_snk-ud-train.iob2", MULTI_MODEL_NAME)

train_dataloader = DataLoader(english_train, shuffle=True, collate_fn=multi_data_collator, batch_size=BATCH_SIZE)
slovak_dataloader = DataLoader(slovak_train, collate_fn=multi_data_collator, batch_size=BATCH_SIZE)

In [ ]:
multi_optimizer = torch.optim.AdamW(multi_model.parameters(), lr=LR)


In [ ]:
multi_model.to(device)

In [20]:
def train_model(model, dataloader, optimizer, epochs, train_whole = True):
    #if not train_whole:
    #  for param in model.bert.parameters():
    #    param.requires_grad = False
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        pbar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Training Epoch {epoch+1}")
        for step, batch in pbar:
            optimizer.zero_grad()
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
        print(total_loss/len(pbar))

In [ ]:
train_model(multi_model, train_dataloader, multi_optimizer, 20)

In [23]:
metric = evaluate.load("seqeval")

In [24]:
def get_labels(predictions, references):
    """
    Convert model outputs (logits) and references (label IDs)
    into human-readable label names, ignoring subword tokens.

    Args:
        predictions: A PyTorch tensor with shape [batch_size, seq_length],
                     containing the predicted label IDs for each token.
        references:  A PyTorch tensor with the true label IDs for each token.

    Returns:
        true_predictions, true_labels:
        - Each is a list of lists of strings.
        - Outer list = batch dimension
        - Inner list = predicted or true labels for each token in that example
        - We skip any token whose label == -100 (these are subword tokens or padding).

    Example:
        Suppose label_list = ["O", "B-PER", "I-PER"],
        predictions = [[0, 1, 2], [0, 0, 1]],
        references  = [[0, 1, 2], [0, 0, -100]]

        Then,
        true_predictions might be [["O", "B-PER", "I-PER"], ["O", "O"]]
        true_labels      might be [["O", "B-PER", "I-PER"], ["O", "O"]]
    """
    predictions = predictions.cpu().numpy()
    references = references.cpu().numpy()
    true_predictions = []
    true_labels = []
    for i, example in enumerate(references):
      true_labels.append([idx_to_tag[idx] for idx in example if idx != -100])
      true_predictions.append([idx_to_tag[idx] for j, idx in enumerate(predictions[i,:]) if references[i, j] != -100])
    return true_predictions, true_labels

In [25]:
def compute_metrics(preds, refs):
    results = metric.compute(predictions=preds, references=refs)
    return {
        "Precision": results["overall_precision"],
        "Recall": results["overall_recall"],
        "F1": results["overall_f1"],
        "Accuracy": results["overall_accuracy"],
    }

In [26]:
# After training, evaluate on the validation set
def eval_model(model, dataloader):
    model.eval()
    validation_progress_bar = tqdm(range(len(dataloader)))
    all_predictions = []
    all_labels = []
    for step, batch in enumerate(dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]
        predicted_labels, true_labels = get_labels(predictions, labels)
        all_predictions.extend(predicted_labels)
        all_labels.extend(true_labels)
        validation_progress_bar.update(1)

    validation_metrics = compute_metrics(all_predictions, all_labels)
    print(validation_metrics)

In [ ]:
eval_model(multi_model, slovak_dataloader)